# SN/ST-07 — Séparation des non résolus avant routage vers les autorités

Même mécanique de routage que SN-06 et ST-06, mais les structures non résolues
sont scindées en deux lots exportés séparément, pour ne pas présenter au
gestionnaire des dossiers rattrapables et des dossiers vides dans le même
classeur.

**Sortie 1 — frôlant le seuil.** Les structures qui ont au moins un candidat
dont le `score_approfondi_ajuste` tombe entre 50 et 66,99, c'est-à-dire juste
sous le seuil de validation. Seuls ces candidats-là sont conservés : si une
structure en a trois et qu'un seul est dans la plage, les deux autres sont
abandonnés — ils ne feraient qu'encombrer l'arbitrage. Le rang est renuméroté
en conséquence, le rang d'origine restant en colonne.

Les structures `NON_FIABLE_APE` rejoignent cette sortie quel que soit leur
score : elles ont un score suffisant et n'ont été écartées que sur la
divergence de code APE. Leur top 3 est conservé entier.

**Sortie 2 — le reste.** Tout ce qui n'entre pas dans la sortie 1, soit les
`DOUTEUX` dont aucun candidat n'atteint 50 et les `REJETE`. Aucun filtrage de
candidat, le top 3 part tel quel.

La partition est exclusive : une structure est dans une sortie ou dans l'autre,
jamais dans les deux. Les deux volets — sirénisation puis siretisation — sont
traités à la suite dans ce notebook, chacun refermant sa connexion.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd

from src import autorites as aut, routage as rt
from src.connexion   import get_finess_connection
from src.chargement  import resoudre_tables, charger_contacts
from src.display     import afficher_tableau
from config.settings import (
    SN_PHASE3, ST_PHASE3, FINESS_EGE_CLEAN, REFERENTIEL_CATEGORIES, ROUTAGE_DIR,
)

## 1. Référentiel des catégories

In [2]:
referentiel = aut.charger_referentiel(REFERENTIEL_CATEGORIES)
print(f"{len(referentiel):,} categories")
print(referentiel["famille"].value_counts().to_string())

181 categories
famille
ARS            135
SOLIDARITES     46


## 2. Règle de partage

Le score de référence est `score_approfondi_ajuste`, celui qui sort de la phase 3
une fois le bonus d'identifiant cohérent appliqué — c'est-à-dire le dernier
calculé. Les deux colonnes de repli ne servent que si un export plus ancien est
rechargé.

In [3]:
SEUIL_BAS, SEUIL_HAUT = 50.0, 67.0          # borne haute exclue : 66,99 passe, 67 non
COLS_SCORE = ["score_approfondi_ajuste", "score_global_approfondi", "score_global"]

STATUT_APE = "NON_FIABLE_APE"


def resoudre_score(df):
    for col in COLS_SCORE:
        if col in df.columns and pd.to_numeric(df[col], errors="coerce").notna().any():
            return col
    raise KeyError(f"aucune colonne de score exploitable parmi {COLS_SCORE}")


def partitionner(df, cle, col_rang="rang"):
    """Sépare les structures frôlant le seuil du reste.

    Renvoie (sortie1, sortie2, recap). Dans la sortie 1 les candidats hors plage
    sont retirés, sauf pour les structures écartées sur l'APE dont le top reste
    entier. Le rang est renumérote pour que chaque structure conserve un rang 1,
    sur lequel s'appuient la synthèse et le décompte de l'export.
    """
    col_score = resoudre_score(df)
    print(f"   score de reference : {col_score}")

    df = df.copy()
    score = pd.to_numeric(df[col_score], errors="coerce")
    dans_plage = score.between(SEUIL_BAS, SEUIL_HAUT, inclusive="left")
    ecarte_ape = df["statut_structure"] == STATUT_APE

    cibles = set(df.loc[dans_plage | ecarte_ape, cle])

    sortie1 = df[df[cle].isin(cibles) & (dans_plage | ecarte_ape)]
    sortie2 = df[~df[cle].isin(cibles)]

    recap = pd.DataFrame([
        {"sortie": "1 - frolant le seuil", "nb_structures": sortie1[cle].nunique(),
         "nb_candidats": len(sortie1),
         "candidats_abandonnes": int((df[cle].isin(cibles)).sum() - len(sortie1))},
        {"sortie": "2 - le reste", "nb_structures": sortie2[cle].nunique(),
         "nb_candidats": len(sortie2), "candidats_abandonnes": 0},
    ])

    return _renumeroter(sortie1, cle, col_rang), _renumeroter(sortie2, cle, col_rang), recap


def _renumeroter(df, cle, col_rang):
    if df.empty:
        return df.assign(rang_origine=pd.Series(dtype="object"))
    df = df.copy()
    df[col_rang] = pd.to_numeric(df[col_rang], errors="coerce")
    df = df.sort_values([cle, col_rang])
    df["rang_origine"] = df[col_rang]
    df[col_rang] = df.groupby(cle).cumcount() + 1
    return df.reset_index(drop=True)

## 2 bis. Bandes de couleur par structure

In [4]:
from pathlib import Path

from openpyxl import load_workbook
from openpyxl.styles import Border, PatternFill, Side

BANDES = ("FFFFFF", "FFF2CC")                        # alternance blanc / jaune pâle
FILET = Side(style="thin", color="D9D9D9")
SEPARATEUR = Side(style="medium", color="A6A6A6")     # frontière entre deux structures


def bander_par_structure(dossier, prefixe, col_cle):
    """Alterne le fond d'une structure à l'autre dans les classeurs déjà écrits.

    La couleur change à chaque changement de valeur de col_cle, et la première
    ligne du groupe reçoit un filet plus marqué. Les lignes d'une même structure
    partagent donc un fond, ce qui la distingue sans avoir à lire son numéro.
    """
    dossier = Path(dossier)
    traites = 0

    for chemin in sorted(dossier.glob(f"{prefixe}_*.xlsx")):
        classeur = load_workbook(chemin)
        touche = False

        for feuille in classeur.worksheets:
            if feuille.title == "Synthèse" or feuille.max_row < 2:
                continue
            colonnes = {c.value: c.column for c in feuille[1]}
            if col_cle not in colonnes:
                continue

            position = colonnes[col_cle]
            precedent, bande = None, 0

            for ligne in feuille.iter_rows(min_row=2, max_row=feuille.max_row):
                valeur = ligne[position - 1].value
                frontiere = precedent is not None and valeur != precedent
                if frontiere:
                    bande = 1 - bande

                fond = PatternFill("solid", fgColor=BANDES[bande])
                haut = SEPARATEUR if frontiere else FILET
                for cellule in ligne:
                    cellule.fill = fond
                    cellule.border = Border(left=FILET, right=FILET,
                                            top=haut, bottom=FILET)
                precedent = valeur

            touche = True

        if touche:
            classeur.save(chemin)
            traites += 1

    return traites

# Volet sirénisation — personnes morales

## 3. Autorité déclarée et coordonnées

In [5]:
CLE = rt.CLE_PM

conn = get_finess_connection()
TABLES = resoudre_tables(conn, {
    "pm": "PM_SMSSE", "ege": "EGE",
    "metadonnee": "Metadonnee", "gestionnaire": "Gestionnaire",
    "telecom": "Telecom",
    "pm_contact": "PmSmsseContact",
})
ae_par_meta = rt.charger_ae_declaree(conn, TABLES)

print("\nCoordonnees :")
contacts = charger_contacts(conn, TABLES, "pm")
conn.close()

   pm           -> T_FAC_PmSmsse
   ege          -> T_FAC_Ege
   metadonnee   -> T_FAC_Metadonnee
   gestionnaire -> T_FAC_Gestionnaire
   telecom      -> T_FAC_Telecom
   pm_contact   -> T_FAC_PmSmsseContact
   158,569 metadonnees rattachees a un gestionnaire, 158,566 avec une autorite exploitable

Coordonnees :
   79,750 PM avec au moins une coordonnee (34,604 mail, 76,534 telephone)


## 4. Chargement et routage

In [6]:
df = rt.charger_phase3(SN_PHASE3, col_cle=CLE)
print(f"{df[CLE].nunique():,} PM, {len(df):,} lignes (top 3 conserve)")
print(df.drop_duplicates(CLE)["statut_structure"].value_counts().to_string())

ege_fils = rt.charger_ege_fils(FINESS_EGE_CLEAN, df["NB_PmSmsseId"].dropna().unique())
print(f"\n{len(ege_fils):,} EGE rattaches a {ege_fils['NB_PmSmsseId'].nunique():,} PM")

resultats = rt.router_sirenisation(df, referentiel, ege_fils, ae_par_meta)
resultats = rt.joindre_contacts(resultats, contacts, "NB_PmSmsseId")

rang1 = resultats[pd.to_numeric(resultats["rang"], errors="coerce") == 1]
print()
print(rang1["autorite_statut"].value_counts().to_string())

2,364 PM, 7,092 lignes (top 3 conserve)
statut_structure
DOUTEUX    2334
REJETE       30

2,086 EGE rattaches a 1,435 PM
   34 structures resolues par l'autorite declaree
   FK_MetadonneeId absente, rattrapage ignore
   TEL_MAIL         2,319 renseignes
   TEL_TELEPHONE    5,439 renseignes
   TEL_TELECOPIE    2,520 renseignes

autorite_statut
RESOLU            1245
NON_DETERMINEE    1105
A_ARBITRER          14


## 5. Partage des deux lots

In [7]:
sn_seuil, sn_reste, recap_sn = partitionner(resultats, CLE)
afficher_tableau(recap_sn, "Sirenisation : repartition des non resolus")

for nom, bloc in [("frolant le seuil", sn_seuil), ("le reste", sn_reste)]:
    if bloc.empty:
        continue
    tete = bloc[pd.to_numeric(bloc["rang"], errors="coerce") == 1]
    print(f"\n{nom} — statut d'origine :")
    print(tete["statut_structure"].value_counts().to_string())

perdu = resultats[CLE].nunique() - (sn_seuil[CLE].nunique() + sn_reste[CLE].nunique())
assert perdu == 0, f"{perdu} PM perdues au partage"

   score de reference : score_approfondi_ajuste


sortie,nb_structures,nb_candidats,candidats_abandonnes
1 - frolant le seuil,1859,4093,1484
2 - le reste,505,1515,0



frolant le seuil — statut d'origine :
statut_structure
DOUTEUX    1859

le reste — statut d'origine :
statut_structure
DOUTEUX    475
REJETE      30


## 6. Export

In [8]:
COLONNES_EXPORT = [
    "autorite_code", "libelle_autorite", "libelle_region", "departement",
    "autorite_statut", "autorite_source", "autorite_candidats", "autorite_motif",
    "ae_declaree_finess", "origine_categorie", "nb_ege_fils", "familles_fils",
    "TX_NumFinessPm", "NB_PmSmsseId", "TX_DenominationPm",
    "adresse_complete_pm", "TEL_MAIL", "TEL_TELEPHONE",
    "TX_Siren", "siren_ref_app", "nom_ul_retenu",
    "adresse_siege_complete_ul", "activitePrincipaleUniteLegale",
    "score_nom", "score_adresse", "score_global", "score_global_approfondi",
    "score_approfondi_ajuste",
    "statut_candidat", "statut_structure", "rang", "rang_origine",
]

LOTS_SN = [
    ("frolant_seuil", "SN_P3_SEUIL", sn_seuil),
    ("hors_seuil",    "SN_P3_RESTE", sn_reste),
]

for dossier, prefixe, bloc in LOTS_SN:
    if bloc.empty:
        print(f"{dossier:<14} : aucun dossier, export ignore")
        continue
    colonnes = [c for c in COLONNES_EXPORT if c in bloc.columns]
    cible = ROUTAGE_DIR / "sirenisation" / dossier
    recap = rt.exporter(bloc, cible, prefixe, col_cle=CLE, colonnes=colonnes)
    attendu, obtenu = bloc[CLE].nunique(), int(recap["nb_structures"].sum())
    assert attendu == obtenu, f"perte au decoupage {dossier} : {attendu} != {obtenu}"
    mis_en_forme = bander_par_structure(cible, prefixe, CLE)
    print(f"{dossier:<14} : {obtenu:,} PM sur {len(recap)} classeurs "
          f"({mis_en_forme} mis en bandes)")

frolant_seuil  : 1,859 PM sur 36 classeurs (36 mis en bandes)
hors_seuil     : 505 PM sur 27 classeurs (27 mis en bandes)


# Volet siretisation — établissements

## 7. Autorité déclarée et coordonnées

In [9]:
del resultats, df, ege_fils, contacts, rang1

CLE = rt.CLE_EGE

conn = get_finess_connection()
TABLES = resoudre_tables(conn, {
    "pm": "PM_SMSSE", "ege": "EGE",
    "metadonnee": "Metadonnee", "gestionnaire": "Gestionnaire",
    "telecom": "Telecom",
    "ege_contact": "EgeContact",
})
ae_par_meta = rt.charger_ae_declaree(conn, TABLES)

print("\nCoordonnees :")
contacts = charger_contacts(conn, TABLES, "ege")
conn.close()

   pm           -> T_FAC_PmSmsse
   ege          -> T_FAC_Ege
   metadonnee   -> T_FAC_Metadonnee
   gestionnaire -> T_FAC_Gestionnaire
   telecom      -> T_FAC_Telecom
   ege_contact  -> T_FAC_EgeContact
   158,569 metadonnees rattachees a un gestionnaire, 158,566 avec une autorite exploitable

Coordonnees :
   150,301 EGE avec au moins une coordonnee (86,666 mail, 145,885 telephone)


## 8. Chargement et routage

In [10]:
df = rt.charger_phase3(ST_PHASE3, col_cle=CLE, col_statut="statut_structure")
print(f"{df[CLE].nunique():,} EGE, {len(df):,} lignes (top 3 conserve)")
print(df.drop_duplicates(CLE)["statut_structure"].value_counts().to_string())

resultats = rt.router_siretisation(df, referentiel, ae_par_meta)
resultats = rt.joindre_contacts(resultats, contacts, "NB_EgeId")

rang1 = resultats[pd.to_numeric(resultats["rang"], errors="coerce") == 1]
print()
print(rang1["autorite_statut"].value_counts().to_string())

10,193 EGE, 30,577 lignes (top 3 conserve)
statut_structure
DOUTEUX           8007
NON_FIABLE_APE    1578
REJETE             608
   FK_MetadonneeId absente, rattrapage ignore
   TEL_MAIL         19,974 renseignes
   TEL_TELEPHONE    24,024 renseignes
   TEL_TELECOPIE    10,365 renseignes

autorite_statut
RESOLU            9979
A_ARBITRER         203
NON_DETERMINEE      11


## 9. Partage des deux lots

In [11]:
st_seuil, st_reste, recap_st = partitionner(resultats, CLE)
afficher_tableau(recap_st, "Siretisation : repartition des non resolus")

for nom, bloc in [("frolant le seuil", st_seuil), ("le reste", st_reste)]:
    if bloc.empty:
        continue
    tete = bloc[pd.to_numeric(bloc["rang"], errors="coerce") == 1]
    print(f"\n{nom} — statut d'origine :")
    print(tete["statut_structure"].value_counts().to_string())

perdu = resultats[CLE].nunique() - (st_seuil[CLE].nunique() + st_reste[CLE].nunique())
assert perdu == 0, f"{perdu} EGE perdus au partage"

   score de reference : score_approfondi_ajuste


sortie,nb_structures,nb_candidats,candidats_abandonnes
1 - frolant le seuil,6451,13312,6041
2 - le reste,3742,11224,0



frolant le seuil — statut d'origine :
statut_structure
DOUTEUX           4873
NON_FIABLE_APE    1578

le reste — statut d'origine :
statut_structure
DOUTEUX    3134
REJETE      608


## 10. Export

In [12]:
COLONNES_EXPORT = [
    "autorite_code", "libelle_autorite", "libelle_region", "departement",
    "autorite_statut", "autorite_source", "autorite_candidats", "autorite_motif",
    "ae_declaree_finess",
    "TX_NumFinessEge", "NB_EgeId", "NB_PmSmsseId", "TX_NomEgeLong",
    "TX_CategorieEntiteGeographiqueExercice", "libelle_categorie", "domaine_categorie",
    "adresse_complete_ege", "TEL_MAIL", "TEL_TELEPHONE",
    "TX_Siret", "siret_ref_app", "nom_etab_retenu",
    "adresse_complete_etab", "activitePrincipaleEtablissement",
    "ape_finess", "ape_candidat", "motif_ape",
    "score_nom", "score_adresse", "score_global", "score_global_approfondi",
    "score_approfondi_ajuste",
    "statut_candidat", "statut_structure", "rang", "rang_origine",
]

LOTS_ST = [
    ("frolant_seuil", "ST_P3_SEUIL", st_seuil),
    ("hors_seuil",    "ST_P3_RESTE", st_reste),
]

for dossier, prefixe, bloc in LOTS_ST:
    if bloc.empty:
        print(f"{dossier:<14} : aucun dossier, export ignore")
        continue
    colonnes = [c for c in COLONNES_EXPORT if c in bloc.columns]
    cible = ROUTAGE_DIR / "siretisation" / dossier
    recap = rt.exporter(bloc, cible, prefixe, col_cle=CLE, colonnes=colonnes)
    attendu, obtenu = bloc[CLE].nunique(), int(recap["nb_structures"].sum())
    assert attendu == obtenu, f"perte au decoupage {dossier} : {attendu} != {obtenu}"
    mis_en_forme = bander_par_structure(cible, prefixe, CLE)
    print(f"{dossier:<14} : {obtenu:,} EGE sur {len(recap)} classeurs "
          f"({mis_en_forme} mis en bandes)")

frolant_seuil  : 6,451 EGE sur 40 classeurs (40 mis en bandes)
hors_seuil     : 3,742 EGE sur 39 classeurs (39 mis en bandes)
